In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto2404")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/especialidades.csv"

In [0]:
especialidades_schema = StructType(fields=[
    StructField("id_especialidad", IntegerType(), True),
    StructField("nombre_especialidad", StringType(), True),
    StructField("area", StringType(), True),
    StructField("costo_consulta_soles", IntegerType(), True),
    StructField("duracion_min", IntegerType(), True)
])

In [0]:
df_especialidades_read = spark.read.option('header', True)\
                                   .schema(especialidades_schema)\
                                   .csv(ruta)

In [0]:
df_especialidades_ingestion = df_especialidades_read.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_especialidades_final = df_especialidades_ingestion.select(
    "id_especialidad",
    "nombre_especialidad",
    "area",
    "costo_consulta_soles",
    "duracion_min",
    "INGESTION_DATE"
)

In [0]:
df_especialidades_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.especialidades")